In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas
import os

(null): No such file or directory
(null): No such file or directory


In [2]:
# initialisation
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
torch.set_default_device('cuda')
print(f"Using {device} device")

g = torch.Generator(device=device)

Using cuda device


(null): No such file or directory
(null): No such file or directory


In [3]:
# data
data = torch.from_numpy(pandas.read_csv('../data/train.csv').to_numpy())

X_train = data[:30000,1:].float()/255
Y_train = data[:30000,0]

X_val = data[30000:36000,1:].float()/255
Y_val = data[30000:36000,0]

X_test = data[36000:42000,1:].float()/255
Y_test = data[36000:42000,0]

In [4]:
# model config
layer_sizes = [784,128,64,32,10]
layer_config = [
  [layer_sizes[0],layer_sizes[1]],
  ['ReLU'],
  [layer_sizes[1],layer_sizes[2]],
  ['ReLU'],
  [layer_sizes[2],layer_sizes[3]],
  ['ReLU'],
  [layer_sizes[3],layer_sizes[4]],
  ['Softmax'],
]

In [5]:
class MNISTDataset(Dataset):
  def __init__(self, X, Y):
    self.X = X
    self.Y = Y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.Y[idx]

In [6]:
train_dataset = MNISTDataset(X_train, Y_train)
cross_val_dataset = MNISTDataset(X_val, Y_val)
test_dataset = MNISTDataset(X_test, Y_test)

train_dataloader = DataLoader(train_dataset, batch_size=100, shuffle=True, generator=g)
cross_val_dataloader = DataLoader(cross_val_dataset, batch_size=100, shuffle=False, generator=g)
test_dataloader = DataLoader(test_dataset, batch_size=100, shuffle=False, generator=g)

In [7]:
class MLP(nn.Module):

  # layers = []
  # layer_sizes = [input, HL1, HL2, ..., output]
  # layer_config = [[layer_size, type], [layer_size, type]]

  def __init__(self, layer_sizes, layer_config):
    super().__init__()
    layers = []
    self.flatten = nn.Flatten()

    # model creation loop
    for entry in layer_config:
      print(entry)

      # hidden layers
      if type(entry[0]) == int:
        layers.append(nn.Linear(entry[0], entry[1]))

      # activation functions
      elif entry[0] == 'ReLU':
        layers.append(nn.ReLU())
      elif entry[0] == 'Softmax':
        pass

    self.network = nn.Sequential(*layers)

  def forward(self, x):
    x = self.flatten(x)
    logits = self.network(x)
    return logits

In [8]:
model = MLP(layer_sizes, layer_config).to(device)
print(model)

[784, 128]
['ReLU']
[128, 64]
['ReLU']
[64, 32]
['ReLU']
[32, 10]
['Softmax']
MLP(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (network): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=10, bias=True)
  )
)


In [9]:
def train(dataloader, model, loss_fn, optimiser):
  size = len(train_dataset)
  model.train()
  for batch, (X,y) in enumerate(train_dataloader):
    X,y = X.to(device), y.to(device)

    # error
    pred = model(X)
    loss = loss_fn(pred,y)

    # back prop
    loss.backward()
    optimiser.step()
    optimiser.zero_grad()

    if batch % 100 == 0:
      loss, current = loss.item(), (batch + 1) * len(X)
      print(f"loss: {loss:>7f}   [{current:>5d}/{size:>5d}]")

In [10]:
def test(dataloader, model, loss_fn):
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  model.eval()
  test_loss, correct = 0,0

  with torch.no_grad():
    for X,y in dataloader:
      X,y = X.to(device), y.to(device)
      pred = model(X)
      test_loss += loss_fn(pred,y).item()
      correct += (pred.argmax(1) == y).type(torch.float).sum().item()
  test_loss /= num_batches
  correct /= size
  print(f"Test Error:\nAccuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f}\n")

In [17]:
loss_fn = nn.CrossEntropyLoss()
optimiser = torch.optim.SGD(model.parameters(), lr=1e-1)
epochs = 100

if os.path.exists('model_checkpoint.pt'):
  checkpoint = torch.load('model_checkpoint.pt')
  model.load_state_dict(checkpoint['model_state_dict'])
  optimiser.load_state_dict(checkpoint['optimiser_state_dict'])
  epoch = checkpoint['epoch'] + 1
  print(f"Resumed from epoch {epoch}")

try:
  for epoch in range(epoch,epochs):
    print(f"Epoch {epoch+1}")
    train(train_dataloader, model, loss_fn, optimiser)
    test(cross_val_dataloader, model, loss_fn)

except (KeyboardInterrupt, Exception) as e:
  print(f"\nstopped early ({type(e).__name__}) - saving parameters\n\t'epoch': {epoch}")
  torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimiser_state_dict': optimiser.state_dict(),
  },'model_checkpoint.pt')
  raise

Resumed from epoch 16
Epoch 17
loss: 0.000124   [  100/30000]
loss: 0.000027   [10100/30000]
loss: 0.000069   [20100/30000]
Test Error:
Accuracy: 97.4%, Avg loss: 0.146240

Epoch 18
loss: 0.000073   [  100/30000]
loss: 0.000109   [10100/30000]
loss: 0.000077   [20100/30000]
Test Error:
Accuracy: 97.5%, Avg loss: 0.146253

Epoch 19
loss: 0.000144   [  100/30000]
loss: 0.000076   [10100/30000]
loss: 0.000205   [20100/30000]
Test Error:
Accuracy: 97.4%, Avg loss: 0.146657

Epoch 20
loss: 0.000074   [  100/30000]
loss: 0.000058   [10100/30000]
loss: 0.000122   [20100/30000]
Test Error:
Accuracy: 97.4%, Avg loss: 0.146591

Epoch 21
loss: 0.000132   [  100/30000]
loss: 0.000100   [10100/30000]
loss: 0.000016   [20100/30000]
Test Error:
Accuracy: 97.4%, Avg loss: 0.146623

Epoch 22
loss: 0.000123   [  100/30000]
loss: 0.000253   [10100/30000]
loss: 0.000096   [20100/30000]
Test Error:
Accuracy: 97.4%, Avg loss: 0.147061

Epoch 23
loss: 0.000093   [  100/30000]
loss: 0.000135   [10100/30000]
l

KeyboardInterrupt: 

In [ ]:
test(test_dataloader, model, loss_fn)

Test Error:
Accuracy: 97.2%, Avg loss: 0.154825



In [ ]:
# save your model!
torch.save(model.state_dict(),"./model.pth")
print("Saved PyTorch Model State to \'model.pth\'!")

Saved PyTorch Model State to 'model.pth'!
